# Prompt Compression

**Step 0**: Install and import the required libraries (`transformers`, `datasets`, `llmlingua`, `torch`)

**Step 1**: Load the text generation model (Llama 3.2) and tokenizer

Model:
- unsloth/Llama-3.2-3B-Instruct

**Step 2**: Define helper functions for text generation and token counting

- `generate_answer()`
- `count_tokens()`

**Step 3**: Load the business earnings call dataset

Dataset:
- Aiera/aiera-ect-sum

**Step 4**: Select an earnings call transcript for the demonstration

- Choose a transcript of suitable length
- Extract the transcript as the context

**Step 5**: Create the prompt

- Define the summarization instruction
- Combine the transcript and the instruction to form the original prompt
- Count the original prompt tokens

**Step 6:** Generate a summary using the original prompt

- Pass the original prompt to Llama.
- Display the generated summary.

**Step 7**: Load the LLMLingua prompt compression model

Model:
- microsoft/llmlingua-2-xlm-roberta-large-meetingbank

**Step 8**: Compress the transcript using LLMLingua

- Compress the context
- Create the compressed prompt by appending the original instruction

**Step 9**: Compare the original and compressed prompts

- Count the original tokens
- Count the compressed tokens
- Calculate the compression percentage

**Step 10**: Generate a summary using the compressed prompt

- Pass the compressed prompt to Llama
- Display the generated summary

**Step 11**: Compare the outputs

- Original prompt length
- Compressed prompt length
- Compression ratio
- Original summary
- Compressed summary
- Observe whether prompt compression preserves the important information while reducing the number of tokens.

**Step 0**: Install and import the required libraries (`transformers`, `datasets`, `llmlingua`, `torch`)

**Step 1**: Load the text generation model (Llama 3.2) and tokenizer

Model:
- unsloth/Llama-3.2-3B-Instruct

In [ ]:
!pip -q install llmlingua datasets rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
import torch
from transformers import pipeline
from llmlingua import PromptCompressor
import pandas as pd
from datasets import load_dataset

In [ ]:
llm = pipeline(
    "text-generation",
    model ="unsloth/Llama-3.2-3B-Instruct",
    dtype= torch.bfloat16,
    device ='cuda',
)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

**Step 2**: Define helper functions for text generation and token counting

- `generate_answer(prompt)`
- `count_tokens()`

    [{"generated_text":                              
        [                                            
            {"role": "system", "content": "..."},    
            {"role": "user", "content": "..."},      
            {"role": "assistant", "content": "..."}  
        ]                                            
    }]  

### Save the tokenizer

In [ ]:
# code here

In [ ]:
def generate_answer(prompt):
  messages = [{
      "role": "user",
      "content": prompt
  }]
  response = llm(messages, max_new_tokens= 300, do_sample= False)
  return response[0]['generated_text'][-1]['content']

In [ ]:
tokenizer = llm.tokenizer

In [ ]:
def count_tokens(text):
  return len(tokenizer(text, add_special_tokens= False)['input_ids'])

In [ ]:
count_tokens("Prompt Compression and Prompt Optimisation")

6

**Step 3**: Load the business earnings call dataset

Dataset:
- Aiera/aiera-ect-sum

Link: https://huggingface.co/datasets/Aiera/aiera-ect-sum

An earnings call transcript is a real conversation between: Company executives (CEO, CFO, etc.),Financial analysts,Investors. These transcripts are often very long, making them ideal for demonstrating prompt compression.
The dataset contains: Earnings call transcripts, Human-written summaries
It was created for summarization research.Each example typically contains:

1.transcript

2.summary

In [ ]:
# code here

In [ ]:
datasets = load_dataset("Aiera/aiera-ect-sum", split="test")
datasets

README.md:   0%|          | 0.00/2.11k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  480kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/38 [00:00<?, ? examples/s]

Dataset({
    features: ['aiera_event_id', 'summary', 'transcript'],
    num_rows: 38
})

**Step 4**: Select an earnings call transcript for the demonstration

- Choose a transcript of suitable length
- Extract the transcript as the context

In [ ]:
# code here

In [ ]:
sample = datasets[3]

In [ ]:
context=  sample['transcript']

In [ ]:
context

"\nJohn T. Williams: Thank you, operator. I'm John T. Williams, Head of Investor Relations. Good afternoon, and thank you for joining us to review Nextdoor's first quarter 2021 financial results. With us on the call today are Nirav Tolia, Executive Chair and incoming Chief Executive Officer; and Matt Anderson, Chief Financial Officer.\nNirav Tolia: Thank you, John T., and good afternoon, everyone. It is an honor to reconnect with you as Nextdoor CEO. Today, I have the same feelings of excitement and possibility as I did when we created this company 14 years ago. Nextdoor has certainly grown a lot since then, but I'm confident that our best times are ahead. We had a productive Q1. But before I get into those details, I'd like to briefly discuss how we're thinking philosophically and practically about taking Nextdoor to the next level. In Silicon Valley, there's a commonly held belief that companies benefit when their founders return. I believe this is due to the value of the founder's m

**Step 5**: Create the prompt

- Define the summarization instruction
- Combine the transcript and the instruction to form the original prompt
- Count the original prompt tokens



### Prompt
question = """ Summarize the earnings call in three bullet points.

1. Revenue reported.
2. Key business drivers.
3. Future guidance. """

In [ ]:
question = """ Summarize the earnings call in three bullet points.
1. Revenue reported.
2. Key business drivers.
3. Future guidance.
"""

In [ ]:
# code here

In [ ]:
original_prompt=  context + "\n\n" + question

In [ ]:
print(original_prompt)


John T. Williams: Thank you, operator. I'm John T. Williams, Head of Investor Relations. Good afternoon, and thank you for joining us to review Nextdoor's first quarter 2021 financial results. With us on the call today are Nirav Tolia, Executive Chair and incoming Chief Executive Officer; and Matt Anderson, Chief Financial Officer.
Nirav Tolia: Thank you, John T., and good afternoon, everyone. It is an honor to reconnect with you as Nextdoor CEO. Today, I have the same feelings of excitement and possibility as I did when we created this company 14 years ago. Nextdoor has certainly grown a lot since then, but I'm confident that our best times are ahead. We had a productive Q1. But before I get into those details, I'd like to briefly discuss how we're thinking philosophically and practically about taking Nextdoor to the next level. In Silicon Valley, there's a commonly held belief that companies benefit when their founders return. I believe this is due to the value of the founder's ment

**Step 6:** Generate a summary using the original prompt

- Pass the original prompt to Llama.
- Display the generated summary.

In [ ]:
# code here

In [ ]:
answer = generate_answer(original_prompt)
print(answer)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here are three bullet points summarizing the earnings call:

• Revenue reported: Nextdoor's Q1 2021 revenue was $53 million, growing 7% year-over-year and 4% sequentially, driven by strong growth in users coming to the platform for the first time.

• Key business drivers: The company's key business drivers include organic verified neighbor growth, weekly active users (WOW) reaching 43.4 million, and a 17 percentage point year-over-year improvement in adjusted EBITDA margin. The self-serve advertising platform is also showing progress, with 100% of self-serve customers using Nextdoor Ads Manager and driving revenue growth.

• Future guidance: Nextdoor expects revenue to be between $229 million and $235 million for the full year 2024, with an expected adjusted EBITDA margin improvement of 15 percentage points year-over-year. The company also expects to generate positive free cash flow in Q4 2024, 12 months ahead of schedule, and has a Q2 2024 outlook of approximately $58 million in reven

**Step 7**: Load the LLMLingua prompt compression model

Model:
- microsoft/llmlingua-2-xlm-roberta-large-meetingbank



In [ ]:
# code here

In [ ]:
compressor = PromptCompressor(
    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
    use_llmlingua2 = True
)

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

**Step 8**: Compress the transcript using LLMLingua

- Compress the context
- Create the compressed prompt by appending the original instruction



In [ ]:
# code here

In [ ]:
compressed = compressor.compress_prompt(context, rate = 0.3)
compressed

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2209 > 512). Running this sequence through the model will result in indexing errors


{'compressed_prompt': "John T. Williams Head of Investor Relations Nextdoor's first quarter 2021 financial results Nirav Tolia Executive Chair Chief Executive Officer Matt Anderson Chief Financial Officer Nirav Tolia honor reconnect Nextdoor CEO excitement possibility 14 years ago Nextdoor grown best times ahead productive Q1 Nextdoor next level Silicon Valley companies benefit founders return founder's mentality Suh Allen founders undervalued business success mindset instill in Nextdoor technologists value from innovative product potential current implementation not fix founders mentality unwavering focus details ownership accountability for results Innovation starts building community build engaged consumer audience attracts customers robust financial results virtuous circle potential business not easy overnight formula successfulenabled creation Nextdoor excited innovative products Q1 promising start momentum organic growth high Weekly active users 43.4 million up 2% year-over-year 

### Prompt
question = """ Summarize the earnings call in three bullet points.

1. Revenue reported.
2. Key business drivers.
3. Future guidance. """

In [ ]:
# code here

In [ ]:
compressed_p= compressed['compressed_prompt']
compressed_prompt =  compressed_p + '\n\n' + question
print(compressed_prompt)

John T. Williams Head of Investor Relations Nextdoor's first quarter 2021 financial results Nirav Tolia Executive Chair Chief Executive Officer Matt Anderson Chief Financial Officer Nirav Tolia honor reconnect Nextdoor CEO excitement possibility 14 years ago Nextdoor grown best times ahead productive Q1 Nextdoor next level Silicon Valley companies benefit founders return founder's mentality Suh Allen founders undervalued business success mindset instill in Nextdoor technologists value from innovative product potential current implementation not fix founders mentality unwavering focus details ownership accountability for results Innovation starts building community build engaged consumer audience attracts customers robust financial results virtuous circle potential business not easy overnight formula successfulenabled creation Nextdoor excited innovative products Q1 promising start momentum organic growth high Weekly active users 43.4 million up 2% year-over-year 4% sequentially engagem

**Step 9**: Compare the original and compressed prompts

- Count the original tokens
- Count the compressed tokens
- Calculate the compression percentage



In [ ]:
# code here

In [ ]:
original_tokens = count_tokens(original_prompt)
compressed_tokens = count_tokens(compressed_prompt)

In [ ]:
print(original_tokens)
print(compressed_tokens)

2004
607


In [ ]:
print(
    "Reduction         :",
    f"{(1-compressed_tokens/original_tokens)*100:.2f}%"
)

Reduction         : 69.71%


**Step 10**: Generate a summary using the compressed prompt

- Pass the compressed prompt to Llama
- Display the generated summary

In [ ]:
# code here

In [ ]:
compressed_answer = generate_answer(compressed_prompt)
print(compressed_answer)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here are three bullet points summarizing the earnings call:

• **Revenue reported**: Nextdoor's Q1 2021 revenue grew 7% year-over-year and 4% sequentially to $53 million, with self-serve customers contributing 50% of the revenue growth.

• **Key business drivers**: The company saw strong engagement, with 43.4 million weekly active users, a 2% year-over-year increase, and a 36% year-over-year increase in content sessions. The advertising platform also showed improvement, with mid-market SMB advertisers spending increasing by 50% and self-serve customers growing their ad spend by 4%.

• **Future guidance**: Nextdoor provided positive guidance for the full year, with EBITDA margin improvement, free cash flow, and a positive outlook for the company's growth. The company also announced a repurchase program, having repurchased 4.4 million shares, and plans to continue investing in its innovative products and technology.


**Step 11**: Compare the outputs

- Original prompt length
- Compressed prompt length
- Compression ratio
- Original summary
- Compressed summary
- Observe whether prompt compression preserves the important information while reducing the number of tokens.

In [ ]:
# code here

In [ ]:
human_summary=sample['summary']
human_summary

'Nextdoor reported Q1 revenue of $53 million, a 7% year-over-year increase driven by strong growth in new verified neighbors joining the platform. Adjusted EBITDA margin improved by 17 percentage points year-over-year, reflecting efficiencies in platform costs, marketing spending, and personnel costs. The company\'s advertising platform showed progress, with self-serve contributing nearly 50% of total Q1 revenue. Looking ahead, Nextdoor expects full-year 2024 revenue between $229-235 million and an adjusted EBITDA margin improvement of approximately 15 percentage points year-over-year. The company raised its adjusted EBITDA guidance and now anticipates generating positive free cash flow in Q4 2024, a year ahead of previous projections. Q2 guidance is for approximately $58 million in revenue and a $13 million adjusted EBITDA loss. Nextdoor is focusing on instilling a "founder\'s mentality" to drive innovation and improvements to its core product, with investments being made in AI capabi

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Original answer compared with itself (reference)
original_rouge = scorer.score(answer,human_summary)["rougeL"].fmeasure

# Compressed answer compared with original answer
compressed_rouge = scorer.score(compressed_answer, human_summary)["rougeL"].fmeasure
comparison = pd.DataFrame({

    "Method": [
        "Original Prompt",
        "LLMLingua"
    ],

    "Prompt Tokens": [
        original_tokens,
        compressed_tokens
    ],

    "Generated Answer": [
        answer,
        compressed_answer
    ],

    "ROUGE-L Score": [
        round(original_rouge, 4),
        round(compressed_rouge, 4)
    ]

})

comparison

,Method,Prompt Tokens,Generated Answer,ROUGE-L Score
0,Original Prompt,2004,Here are three bullet points summarizing the e...,0.3802
1,LLMLingua,607,Here are three bullet points summarizing the e...,0.2297


# Prompt Optimisation and Compression

### Loading LLM and defining utility functions
import transformers and torch

In [ ]:
from transformers import pipeline
import torch

In [ ]:
# load the LLM

In [ ]:
llm = pipeline(
    "text-generation",
    model ="unsloth/Llama-3.2-3B-Instruct",
    dtype= torch.bfloat16,
    device ='cuda',
)

In [ ]:
def make_message(system_content, user_content):
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def fetch_response(response):
    return response[0]["generated_text"][-1]["content"]

In [ ]:
def generate(system_prompt, user_prompt, max_new_tokens =300):
  messages = make_message(system_prompt, user_prompt)
  response = llm(messages, max_new_tokens= max_new_tokens, do_sample= False)
  return fetch_response(response)

## Prompt Optmization

We will use the **ProTeGi** automatic prompt optimization algorithm for this demonstration. ProTeGi uses textual gradients to find flaws in the initial prompt and uses another (or same) LLM to create optimized version of the prompt.

**Scenario**: An application for a loan has been received by the bank. They have to decide whether to accept or reject the loan based upon certain lending criteria and parameters about the applicant.

It is difficult to manually write a prompt to automate this process. It is also difficult to manually optimize a simple prompt for such a complex scenario. Automatic prompt optimization is useful in this case to derive an optimized prompt starting from a simple prompt.

#### **Step 0:** Background info

In [ ]:
credit_policy = """
NORTHBRIDGE BANK - RETAIL LENDING POLICY

P1. Debt-to-Income Ratio (DTI) = Existing Debt / Annual Income.
    DTI must be less than 0.40.

P2. Loan-to-Income Ratio (LTI) = Requested Loan / Annual Income.
    LTI must not exceed 3.0.

P3. Minimum credit score is 660.
    Scores between 620 and 659 are acceptable ONLY if collateral is provided.

P4. Applicants must have at least 12 months of continuous employment.

P5. Unsecured loans above $100,000 are not permitted.

P6. Final decision must be one of:
    APPROVE
    REJECT
    CONDITIONAL APPROVAL
"""


In [ ]:
loan_application = """
Applicant Name: John Smith

Age: 28

Annual Income: $48,000

Employment:
Software Developer at ABC Technologies

Employment Duration:
10 months

Credit Score:
615

Existing Debt:
$22,000

Requested Loan Amount:
$180,000

Loan Purpose:
Purchase a house

Collateral:
None

Previous Loan Defaults:
None
"""


For this applicant,\
DTI = 22000 / 48000\
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= 0.458 (violate P1)

LTI = 180000 / 48000\
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= 3.75 (violate P2)

Applicant Score is 615 with no collateral (violate P3)

Employment Duration is 10 months (violate P4)

Applicant request of loan 180000 with no collateral (violate p5)

**Correct decision is reject**

In [ ]:
correct_decision = "reject"

#### **Step 1:** Define the Initial Prompt

In [ ]:
system_prompt = """
You are a bank loan officer.

Review the loan application and decide whether the loan should be approved.

Provide a brief explanation.
"""

user_prompt = f"""
BANK LENDING POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}
"""

## Step 2 : Generate the initial answer

In [ ]:
# generate the initial answer


In [ ]:
initial_answer = generate(system_prompt, user_prompt, max_new_tokens = 300)
print(initial_answer)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the provided loan application and Northbridge Bank's retail lending policy, I am going to approve the loan with a conditional approval.

Here's the explanation:

- The applicant's debt-to-income ratio (DTI) is $22,000 / $48,000 = 0.46, which exceeds the maximum allowed limit of 0.40. However, this is not a deal-breaker, as the applicant is requesting a relatively high loan amount.

- The loan-to-income ratio (LTI) is $180,000 / $48,000 = 3.75, which exceeds the maximum allowed limit of 3.0. This is a significant concern, but the applicant's income is relatively low, which may justify the higher loan amount.

- The applicant's credit score is 615, which falls within the acceptable range of 620-659 when collateral is provided. However, since no collateral is offered, this is a concern.

- The applicant has less than 12 months of continuous employment, which does not meet the minimum requirement of 12 months.

- The requested loan amount is above $100,000, which is not permitted 

#### **Step 3:** Evaluate the Response
- Use an **LLM-as-a-Judge** (can be same or different model) to evaluate the generated response.
- The judge assigns a score and identifies:
  - Correct reasoning
  - Missing information
  - Errors or hallucinations
  - Policy violations

In [ ]:
judge_system_prompt = """
You are an expert loan approval auditor.

Evaluate ONLY the quality of the model's reasoning.
Use the following rubric (10 points).
1. Correctly evaluates the credit score. (1 point)
2. Correctly evaluates annual income. (1 point)
3. Correctly evaluates existing debt. (1 point)
4. Correctly evaluates employment stability. (1 point)
5. Correctly evaluates collateral. (1 point)
6. Computes or discusses the Debt-to-Income (DTI) ratio. (1 point)
7. Computes or discusses the Loan-to-Income (LTI) ratio. (1 point)
8. References the lending policy when making the decision. (1 point)
9. Gives a clear evidence-based justification. (1 point)
10. Correct final decision. (1 point)

Be STRICT.
Only award a point if the criterion is explicitly satisfied.
A generic explanation should score between 3 and 5.
A detailed policy-based analysis should score between 8 and 10.

Return EXACTLY in this format:

Score: X/10
Strengths:
- ...
Missing:
- ...
"""

judge_user_prompt = f"""
BANK POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}

----------------------------------------

MODEL RESPONSE

{initial_answer}

----------------------------------------

ACTUAL FINAL DECISION

{correct_decision}
"""

In [ ]:
# genrate the judge feedback


In [ ]:
judge_feedback = generate(judge_system_prompt, judge_user_prompt, max_new_tokens = 300)
print(judge_feedback)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: 4/10
Strengths:
- Correctly evaluates the debt-to-income ratio and loan-to-income ratio.
- Correctly evaluates the employment duration.
- Correctly evaluates the requested loan amount.
Missing:
- Does not correctly evaluate the credit score (fails to mention that it falls within the acceptable range when collateral is provided).
- Does not correctly evaluate the collateral (fails to mention that no collateral is provided).
- Does not reference the lending policy when making the decision (fails to mention that the requested loan amount exceeds the policy limit).
- Does not give a clear evidence-based justification (fails to provide a clear explanation for the conditional approval).
- Does not correctly compute or discuss the DTI and LTI ratios (fails to mention that the DTI ratio exceeds the maximum allowed limit).
- Does not correctly compute or discuss the LTI ratio (fails to mention that the LTI ratio exceeds the maximum allowed limit).
- Does not provide a clear final decisio

### **Step 4:** Generate a Textual Gradient
Generate a **textual gradient**, i.e., natural-language feedback describing how the prompt should be improved.

In [ ]:
gradient_system_prompt = """
Your task is to find out the textual gradients of a prompt. The textual gradient is the analysis of why the CURRENT PROMPT produced a weak response.

Do NOT rewrite the prompt.

Instead, identify the instructions that are missing from the prompt.

Focus on:

- Missing reasoning steps
- Missing financial analyses
- Missing policy references
- Missing output structure
- Missing justification requirements

Write 4-6 concise bullet points.

Each bullet should explain:

- what the prompt failed to instruct
- why that caused a weaker answer

Return ONLY the textual gradient.
"""

gradient_user_prompt = f"""
CURRENT PROMPT

{system_prompt}

--------------------------------------------------

BANK POLICY

{credit_policy}

--------------------------------------------------

LOAN APPLICATION

{loan_application}

--------------------------------------------------

MODEL RESPONSE

{initial_answer}

--------------------------------------------------

JUDGE FEEDBACK

{judge_feedback}

Generate the textual gradient.
"""

In [ ]:
# generate the textual_gradient

In [ ]:
textual_gradient = generate(gradient_system_prompt, gradient_user_prompt, max_new_tokens = 300)
print(textual_gradient)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here are the textual gradients:

• The prompt failed to instruct on how to handle cases where the applicant's credit score falls within the acceptable range when collateral is provided, leading to a weaker answer.

• The prompt failed to instruct on how to evaluate the collateral provided by the applicant, resulting in a failure to mention that no collateral is provided.

• The prompt failed to instruct on how to reference the lending policy when making the decision, leading to a failure to mention that the requested loan amount exceeds the policy limit.

• The prompt failed to instruct on how to provide a clear evidence-based justification for the conditional approval, resulting in a vague explanation.

• The prompt failed to instruct on how to correctly compute or discuss the debt-to-income ratio and loan-to-income ratio, leading to a failure to mention that the DTI ratio exceeds the maximum allowed limit.

• The prompt failed to instruct on how to provide a clear final decision, res

### **Step 5:** Optimize the Prompt

In [ ]:
optimizer_system_prompt = """
You are an expert Prompt Engineer. Your task is to improve the prompt using ONLY the textual gradient.

Requirements:

- Preserve the original task.
- Incorporate the feedback from the textual gradient.
- Add only the missing instructions.
- Return ONLY the improved prompt.
"""

optimizer_user_prompt = f"""
ORIGINAL PROMPT

{system_prompt}

--------------------------------------------------

TEXTUAL GRADIENT

{textual_gradient}

--------------------------------------------------

Rewrite the prompt by incorporating the textual gradient.
"""

In [ ]:
# generate the optimised prompt

In [ ]:
optimised_prompt = generate(optimizer_system_prompt, optimizer_user_prompt, max_new_tokens = 300)
print(optimised_prompt)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are a bank loan officer.

Review the loan application and decide whether the loan should be approved, considering the following factors:

1. Evaluate the applicant's credit score and determine if it falls within the acceptable range. If the credit score is within the acceptable range, consider the collateral provided by the applicant. If collateral is provided, assess its value and determine if it meets the minimum requirements to offset the loan amount. If collateral is not provided, explain why it was not submitted.

2. Reference the lending policy to ensure that the requested loan amount does not exceed the policy limit. If the requested loan amount exceeds the policy limit, explain why the loan is not approved and provide a clear justification for the decision.

3. Compute or discuss the debt-to-income ratio and loan-to-income ratio to ensure that they are within the acceptable limits. If the DTI ratio exceeds the maximum allowed limit, explain why the loan is not approved and 

### **Step 6:** Generate the Optimized Response
Generate a new response using the optimized prompt for the same input.

In [ ]:
# generate the optimised answer


In [ ]:
optmised_answer = generate(optimised_prompt, user_prompt, max_new_tokens = 400)
print(optmised_answer)

[transformers] Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Loan Application Review and Decision**

**Applicant Information:**

* Name: John Smith
* Age: 28
* Annual Income: $48,000
* Employment: Software Developer at ABC Technologies
* Employment Duration: 10 months
* Credit Score: 615
* Existing Debt: $22,000
* Requested Loan Amount: $180,000
* Loan Purpose: Purchase a house
* Collateral: None
* Previous Loan Defaults: None

**Policy Evaluation:**

1. **Credit Score:** The applicant's credit score is 615, which falls within the acceptable range of 620-659 if collateral is provided. However, since no collateral is provided, the credit score is not within the acceptable range. Therefore, the loan application is not approved solely based on the credit score.

2. **Debt-to-Income Ratio (DTI):** The applicant's DTI ratio is calculated as $22,000 (existing debt) / $48,000 (annual income) = 0.46, which exceeds the maximum allowed limit of 0.40. Therefore, the loan application is not approved solely based on the DTI ratio.

3. **Loan-to-Income Rati

### Although the model has correctly evaluated the bank's lending policy, it still gives the wrong decision.